In [ ]:
from flask import Flask, request, jsonify, render_template, redirect, url_for, abort
import sqlite3
import string, random
from datetime import datetime, timezone

app = Flask(__name__)
DB = 'urls.db'

def init_db():
    with sqlite3.connect(DB) as conn:
        c = conn.cursor()
        c.execute('''
            CREATE TABLE IF NOT EXISTS urls (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                url TEXT NOT NULL,
                shortCode TEXT UNIQUE NOT NULL,
                createdAt TEXT DEFAULT CURRENT_TIMESTAMP,
                updatedAt TEXT DEFAULT CURRENT_TIMESTAMP,
                accessCount INTEGER DEFAULT 0
            )
        ''')
        conn.commit()

def generate_short_code(length=6):
    return ''.join(random.choices(string.ascii_letters + string.digits, k=length))

@app.route('/')
def home():
    msg = request.args.get('msg')
    code = request.args.get('code')
    with sqlite3.connect(DB) as conn:
        c = conn.cursor()
        c.execute('SELECT id, url, shortCode, createdAt, updatedAt, accessCount FROM urls ORDER BY id')
        urls = c.fetchall()
    return render_template('index.html', urls=urls, msg=msg, code=code)

@app.route('/r/<shortCode>')
def redirect_to_original(shortCode):
    shortCode = ''.join(e for e in shortCode if e.isalnum())
    with sqlite3.connect(DB) as conn:
        c = conn.cursor()
        c.execute('SELECT url FROM urls WHERE shortCode = ?', (shortCode,))
        row = c.fetchone()
        if row:
            c.execute('UPDATE urls SET accessCount = accessCount + 1 WHERE shortCode = ?', (shortCode,))
            conn.commit()
            return redirect(row[0])
        else:
            abort(404)

# 👉 New route: when user clicks original URL, increment count
@app.route('/go/<int:url_id>')
def go_to_original(url_id):
    with sqlite3.connect(DB) as conn:
        c = conn.cursor()
        c.execute('SELECT url FROM urls WHERE id = ?', (url_id,))
        row = c.fetchone()
        if row:
            c.execute('UPDATE urls SET accessCount = accessCount + 1 WHERE id = ?', (url_id,))
            conn.commit()
            return redirect(row[0])
        else:
            abort(404)

@app.route('/shorten', methods=['POST'])
def create_short_url():
    data = request.form.get('url') or (request.json and request.json.get('url'))
    if not data:
        return jsonify({'error': 'Missing url'}), 400
    original_url = data
    custom = request.form.get('customCode')
    if custom:
        short_code = ''.join(e for e in custom if e.isalnum())
    else:
        short_code = generate_short_code()
    now = datetime.now(timezone.utc).isoformat()
    try:
        with sqlite3.connect(DB) as conn:
            c = conn.cursor()
            c.execute('INSERT INTO urls (url, shortCode, createdAt, updatedAt) VALUES (?, ?, ?, ?)',
                      (original_url, short_code, now, now))
            conn.commit()
        return redirect(url_for('home', msg='Created successfully', code=201))
    except sqlite3.IntegrityError:
        return redirect(url_for('home', msg='Short code already exists', code=400))

@app.route('/delete/<shortCode>', methods=['POST'])
def delete_short_url_form(shortCode):
    shortCode = ''.join(e for e in shortCode if e.isalnum())
    with sqlite3.connect(DB) as conn:
        c = conn.cursor()
        c.execute('DELETE FROM urls WHERE shortCode = ?', (shortCode,))
        if c.rowcount == 0:
            return redirect(url_for('home', msg='Short URL not found', code=404))
        conn.commit()
    return redirect(url_for('home', msg='Deleted successfully', code=204))

@app.route('/update/<shortCode>', methods=['POST'])
def update_short_url_form(shortCode):
    shortCode = ''.join(e for e in shortCode if e.isalnum())
    new_url = request.form.get('new_url')
    new_short = request.form.get('new_shortCode')
    updatedAt = datetime.now(timezone.utc).isoformat()
    if not new_url and not new_short:
        return redirect(url_for('home', msg='Provide at least URL or new short code', code=400))
    with sqlite3.connect(DB) as conn:
        c = conn.cursor()
        if new_url and new_short:
            new_short = ''.join(e for e in new_short if e.isalnum())
            c.execute('UPDATE urls SET url = ?, shortCode = ?, updatedAt = ? WHERE shortCode = ?',
                      (new_url, new_short, updatedAt, shortCode))
        elif new_url:
            c.execute('UPDATE urls SET url = ?, updatedAt = ? WHERE shortCode = ?',
                      (new_url, updatedAt, shortCode))
        elif new_short:
            new_short = ''.join(e for e in new_short if e.isalnum())
            c.execute('UPDATE urls SET shortCode = ?, updatedAt = ? WHERE shortCode = ?',
                      (new_short, updatedAt, shortCode))
        if c.rowcount == 0:
            return redirect(url_for('home', msg='Short URL not found', code=404))
        conn.commit()
    return redirect(url_for('home', msg='Updated successfully', code=200))

# ✅ Run once to reset IDs (delete all & reset counter)
@app.route('/reset_ids')
def reset_ids():
    with sqlite3.connect(DB) as conn:
        c = conn.cursor()
        c.execute('DELETE FROM urls')
        c.execute('DELETE FROM sqlite_sequence WHERE name="urls"')
        conn.commit()
    return redirect(url_for('home', msg='All data deleted and IDs reset', code=200))

if __name__ == '__main__':
    init_db()
    app.run(port=5000, debug=True, use_reloader=False)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
